# Phase 4 — Foundation Model (Chronos)

Three stages:
1. **Zero-shot**: load pretrained Chronos, predict with no training — does it beat XGBoost?
2. **Fine-tune**: continue training on OPSD series, improve on domain-specific patterns
3. **Leaderboard**: unified comparison of all models with uncertainty intervals

Install first:
```bash
pip install torch chronos-forecasting transformers accelerate
```

In [ ]:
import sys, warnings, math
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

PROCESSED = Path('../data/processed')
RESULTS   = Path('../results')
MODELS    = Path('../models')
for d in [RESULTS, MODELS, RESULTS/'leaderboard_plots', MODELS/'chronos_finetuned']:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi'] = 110

device = 'cuda' if torch.cuda.is_available() else \
         'mps'  if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print('Setup complete.')

In [ ]:
from src.features.pipeline import build_features, get_feature_cols, TARGET_COLS
from src.features.scaling import split_and_scale
from src.models.chronos_model import ChronosForecaster
from src.models.metrics import MetricResult, ResultsRegistry, eval_by_horizon

df = build_features(PROCESSED)
feat_cols = get_feature_cols(df)

splits = split_and_scale(
    df, target_cols=TARGET_COLS, feature_cols=feat_cols,
    train_end='2021-12-31', val_end='2022-12-31',
)
train, val, test = splits['train'], splits['val'], splits['test']
registry = ResultsRegistry(RESULTS / 'all_metrics.jsonl')

HORIZON = 24
print(f'Train: {len(train):,}  Val: {len(val):,}  Test: {len(test):,}')
# Pipeline now tracks 9 targets (load, carbon intensity, and 7 generation-mix series)
print(f'Targets ({len(TARGET_COLS)}): {TARGET_COLS}')

## 1. Zero-shot Chronos — no training needed

Load `chronos-t5-small` and predict directly on the test set.
Context window = 168 h (1 week) per forecast origin.

**Warmup**: `val` is passed as `history_df` so Chronos can build its
recurrent state before the test window begins — avoiding cold-start amnesia.

**Tip**: On CPU this takes ~2 min for 500 rows. Pass `test.iloc[:200]` for a quick smoke test.

In [ ]:
# Quick smoke-test: first 200 rows of test set
# Remove the slice for full evaluation
TEST_SLICE = test.iloc[:200]

zs_models = {}
for target in TARGET_COLS:
    print(f'\n-- Chronos zero-shot: {target} --')
    model = ChronosForecaster(
        target_col=target,
        horizon=HORIZON,
        model_id='amazon/chronos-t5-small',
        context_length=168,
        num_samples=20,
        device=device,
    )
    # Pass val as history_df so the model has warmup context at the test boundary
    result = model.evaluate(TEST_SLICE, split_name='test', history_df=val)
    registry.add(result)
    zs_models[target] = model
    print(result)

print('\nZero-shot evaluation complete.')

## 2. Zero-shot vs XGBoost — head-to-head on load_mw

For Chronos, the validation set is prepended before calling `.predict()` so
that the model has a full lookback window at position 0 of the test slice.
The prediction array is then sliced back to `TEST_SLICE` length.

In [ ]:
try:
    from src.models.xgboost_model import XGBoostForecaster
    target = 'load_mw'
    N = min(168, len(TEST_SLICE))

    xgb = XGBoostForecaster.load(MODELS / 'baselines' / f'xgboost_{target}')
    xgb_preds = xgb.predict(TEST_SLICE)[:N, 0]

    # Prepend val so Chronos has a filled context window from the first test step
    combined_zs = pd.concat([val, TEST_SLICE])
    zs_preds = zs_models[target].predict(combined_zs)[len(val):len(val) + N, 0]

    actual = TEST_SLICE[target].values[:N]

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(actual,    lw=2.0, label='Actual',            color='steelblue')
    ax.plot(xgb_preds, lw=1.5, label='XGBoost h=1',      color='crimson',    alpha=0.85)
    ax.plot(zs_preds,  lw=1.3, label='Chronos zero-shot', color='darkorange', alpha=0.85, ls='--')
    ax.set_title(f'{target}: XGBoost vs Chronos zero-shot')
    ax.set_xlabel('Hour'); ax.set_ylabel('MW'); ax.legend()
    plt.tight_layout()
    plt.savefig(RESULTS / 'leaderboard_plots' / 'zs_vs_xgb_load.png', bbox_inches='tight')
    plt.show()
except Exception as e:
    print(f'XGBoost checkpoint not found or error: {e}')
    print('Run 03_baselines.ipynb first to train XGBoost.')

## 3. Uncertainty intervals — Chronos probabilistic output

Chronos returns full sample distributions. Here we plot the 10th–90th
percentile prediction interval alongside the median forecast.

**Warmup**: `val` is concatenated with `sample_df` before calling
`predict_quantiles()`, and the returned arrays are sliced back to the
original `sample_df` length so the plot covers only the 72-h test window.

In [ ]:
target = 'load_mw'
model  = ChronosForecaster(
    target_col=target, horizon=HORIZON,
    model_id='amazon/chronos-t5-small',
    context_length=168, num_samples=100,
    quantile_levels=[0.1, 0.5, 0.9],
    device=device,
)

# Use first 72 h of test (3 days) for a clean plot
sample_df = TEST_SLICE.iloc[:72]

# Prepend val to give Chronos a warm context window; slice results back to sample_df
combined      = pd.concat([val, sample_df])
quantiles_all = model.predict_quantiles(combined)
quantiles     = {q: arr[len(val):] for q, arr in quantiles_all.items()}

actual = sample_df[target].values
hours  = np.arange(len(actual))
h1     = 0   # h=1 slice across all origins

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(
    hours,
    quantiles[0.1][:, h1],
    quantiles[0.9][:, h1],
    alpha=0.25, color='darkorange', label='10–90% interval'
)
ax.plot(hours, quantiles[0.5][:, h1], color='darkorange', lw=1.5, label='Median (h=1)')
ax.plot(hours, actual,                color='steelblue',  lw=2.0, label='Actual')
ax.set_title(f'{target}: Chronos probabilistic forecast (72 h, h=1 slice)')
ax.set_xlabel('Hour'); ax.set_ylabel('MW'); ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / 'leaderboard_plots' / 'chronos_uncertainty.png', bbox_inches='tight')
plt.show()

## 4. Fine-tune Chronos on OPSD data

Continue training the pretrained model on your target series.
- 10 epochs, cosine LR, early stopping on val NLL
- GPU: ~5 min per target. CPU: ~30 min (reduce `num_epochs` to 3 for a quick test)

Set `RUN_FINETUNE = True` to execute.

In [ ]:
RUN_FINETUNE = False   # <-- flip to True when ready

ft_models = {}

if RUN_FINETUNE:
    for target in TARGET_COLS:
        print(f'\n-- Fine-tuning Chronos: {target} --')
        model = ChronosForecaster(
            target_col=target,
            horizon=HORIZON,
            model_id='amazon/chronos-t5-small',
            context_length=168,
            num_samples=20,
            device=device,
            finetune_config={
                'learning_rate': 1e-4,
                'num_epochs':    10,     # reduce to 3 for CPU speed test
                'batch_size':    32,
                'early_stopping_patience': 3,
            },
        )
        model.fit(train, val_df=val)
        model.save(MODELS / 'chronos_finetuned' / target)
        ft_models[target] = model

        # Pass val as history_df so the model warms up before the test boundary
        result = model.evaluate(TEST_SLICE, split_name='test', history_df=val)
        registry.add(result)
        print(result)

    print('\nFine-tuning complete.')
else:
    print('RUN_FINETUNE = False. Set to True to execute fine-tuning.')

## 5. Fine-tuned vs zero-shot vs XGBoost — horizon curves

For any Chronos variant, `val` is prepended to `TEST_SLICE` before calling
`.predict()`, and the returned rows are sliced back to `TEST_SLICE` length
so MAE is computed only on true test-set positions.

In [ ]:
if RUN_FINETUNE and ft_models:
    target    = 'load_mw'
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    models_to_compare = {
        'Chronos ZS': zs_models[target],
        'Chronos FT': ft_models[target],
    }
    try:
        from src.models.xgboost_model import XGBoostForecaster
        models_to_compare['XGBoost'] = XGBoostForecaster.load(
            MODELS / 'baselines' / f'xgboost_{target}'
        )
    except Exception:
        pass

    colors = {'Chronos ZS': 'darkorange', 'Chronos FT': 'crimson', 'XGBoost': 'steelblue'}
    y_true = TEST_SLICE[target].values
    N      = min(168, len(TEST_SLICE))

    for label, m in models_to_compare.items():
        # Chronos needs warmup history; XGBoost is stateless so predict on TEST_SLICE directly
        if 'Chronos' in label:
            combined = pd.concat([val, TEST_SLICE])
            preds    = m.predict(combined)[len(val):]
        else:
            preds = m.predict(TEST_SLICE)

        maes = []
        for h in range(HORIZON):
            yp = preds[:, h]
            if h == 0:
                yt = y_true.copy()
            else:
                yt = np.full_like(y_true, np.nan); yt[:-h] = y_true[h:]
            mask = np.isfinite(yt) & np.isfinite(yp)
            maes.append(float(np.mean(np.abs(yt[mask] - yp[mask]))))
        axes[0].plot(range(1, HORIZON + 1), maes, label=label,
                     color=colors.get(label, 'gray'), marker='o', ms=3)

    axes[0].set_title(f'{target} — MAE by horizon'); axes[0].set_xlabel('h')
    axes[0].set_ylabel('MAE [MW]'); axes[0].legend()
    axes[0].set_xticks([1, 6, 12, 18, 24])

    # 7-day forecast panel
    axes[1].plot(y_true[:N], lw=2, label='Actual', color='steelblue')
    for label, m in models_to_compare.items():
        if 'Chronos' in label:
            combined = pd.concat([val, TEST_SLICE])
            p = m.predict(combined)[len(val):len(val) + N, 0]
        else:
            p = m.predict(TEST_SLICE)[:N, 0]
        axes[1].plot(p, lw=1.3, alpha=0.8, label=label,
                     color=colors.get(label, 'gray'), ls='--')
    axes[1].set_title(f'{target} — 7-day forecast'); axes[1].set_xlabel('Hour')
    axes[1].legend(fontsize=8)

    plt.suptitle('Chronos fine-tuned vs zero-shot vs XGBoost', fontsize=12)
    plt.tight_layout()
    plt.savefig(RESULTS / 'leaderboard_plots' / 'ft_vs_zs_vs_xgb.png', bbox_inches='tight')
    plt.show()
else:
    print('Fine-tuning not run yet. Set RUN_FINETUNE=True in cell 4.')

## 6. Full leaderboard — all models ranked

In [ ]:
results_df   = registry.to_dataframe()
test_results = results_df[results_df['split'] == 'test'].copy()

if test_results.empty:
    print('No results yet — run the evaluation cells above first.')
else:
    board = test_results.sort_values(['target', 'mae'])

    print('\n=== FULL LEADERBOARD (test set, sorted by MAE) ===')
    print(f'{"MODEL":<24} {"TARGET":<22} {"MAE":>8} {"RMSE":>8} {"MAPE%":>7} {"R2":>7}')
    print('-' * 74)
    prev = None
    for _, row in board.iterrows():
        if row['target'] != prev and prev is not None:
            print()
        prev = row['target']
        print(f"{row['model']:<24} {row['target']:<22} "
              f"{row['mae']:>8.1f} {row['rmse']:>8.1f} "
              f"{row.get('mape', float('nan')):>7.2f} "
              f"{row.get('r2',   float('nan')):>7.4f}")

    board.to_csv(RESULTS / 'leaderboard.csv', index=False)
    print(f'\nSaved -> {RESULTS / "leaderboard.csv"}')

## 7. Leaderboard bar chart — visual comparison

The subplot grid is built dynamically from `TARGET_COLS` (currently 9 targets)
so it stays correct if the target list grows or shrinks. Unused subplot slots
are hidden automatically.

In [ ]:
if not test_results.empty:
    color_map = {
        'xgboost':    '#1D9E75',
        'sarima':     '#BA7517',
        'chronos-zs': '#7F77DD',
        'chronos-ft': '#D85A30',
    }

    # Dynamic grid — 2 columns, enough rows for all targets
    cols = 2
    rows = math.ceil(len(TARGET_COLS) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))

    for ax, target in zip(axes.flat, TARGET_COLS):
        sub    = test_results[test_results['target'] == target].sort_values('mae')
        colors = [color_map.get(m, '#888780') for m in sub['model']]
        bars   = ax.barh(sub['model'], sub['mae'], color=colors, height=0.55)
        ax.set_xlabel('MAE (lower is better)')
        ax.set_title(target)
        ax.invert_yaxis()
        for bar in bars:
            w = bar.get_width()
            ax.text(w * 1.01, bar.get_y() + bar.get_height() / 2,
                    f'{w:.1f}', va='center', fontsize=9)

    # Hide any unfilled subplot slots
    for j in range(len(TARGET_COLS), len(axes.flat)):
        axes.flat[j].set_visible(False)

    legend_handles = [
        mpatches.Patch(color='#1D9E75', label='XGBoost (baseline)'),
        mpatches.Patch(color='#BA7517', label='SARIMA (baseline)'),
        mpatches.Patch(color='#7F77DD', label='Chronos zero-shot'),
        mpatches.Patch(color='#D85A30', label='Chronos fine-tuned'),
    ]
    fig.legend(handles=legend_handles, loc='lower center', ncol=4,
               bbox_to_anchor=(0.5, -0.02), fontsize=10)

    plt.suptitle(
        f'Phase 4 leaderboard — all models, test set MAE ({len(TARGET_COLS)} targets)',
        fontsize=13
    )
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(RESULTS / 'leaderboard_plots' / 'leaderboard_bar.png', bbox_inches='tight')
    plt.show()

## 8. Context length ablation — how much history does Chronos need?

Sweep context_length ∈ {24, 72, 168, 336} and compare MAE on load_mw.
This is a quick experiment to justify (or challenge) the 168h default.

`val` is prepended in every sweep iteration so each context length is
evaluated under identical warmup conditions.

In [ ]:
RUN_ABLATION = False   # flip to True — takes ~10 min on CPU

if RUN_ABLATION:
    target   = 'load_mw'
    contexts = [24, 72, 168, 336]
    results  = []
    combined_abl = pd.concat([val, TEST_SLICE])

    for ctx in contexts:
        print(f'context_length={ctx} ...')
        m = ChronosForecaster(
            target_col=target, horizon=HORIZON,
            model_id='amazon/chronos-t5-small',
            context_length=ctx, num_samples=20, device=device,
        )
        # Slice back to TEST_SLICE rows after warmup from val
        preds  = m.predict(combined_abl)[len(val):]
        y_true = TEST_SLICE[target].values
        mask   = np.isfinite(y_true) & np.isfinite(preds[:, 0])
        mae    = float(np.mean(np.abs(y_true[mask] - preds[mask, 0])))
        results.append({'context_length': ctx, 'mae': mae})
        print(f'  MAE = {mae:.2f} MW')

    abl_df = pd.DataFrame(results)
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(abl_df['context_length'], abl_df['mae'],
            marker='o', ms=6, color='#7F77DD', lw=2)
    ax.set_xlabel('Context length (hours)')
    ax.set_ylabel('MAE [MW]')
    ax.set_title(f'{target} — Chronos zero-shot: context length ablation')
    ax.set_xticks(contexts)
    plt.tight_layout()
    plt.savefig(RESULTS / 'leaderboard_plots' / 'context_ablation.png', bbox_inches='tight')
    plt.show()
else:
    print('Set RUN_ABLATION=True to run context length sweep.')